# GNN ClusterGCN (Multi-PNA-EU) — Colab 실행 노트북

**baseline과 달라진 점**
- 샘플링 방식: `LinkNeighborLoader` → `ClusterData + ClusterLoader` (stochastic multiple partitions)
- 새 하이퍼파라미터: `num_parts`(METIS 파티션 수), `clusters_per_batch`(배치당 파티션 수)
- `batch_size`, `num_neighs`, `weighted_sampler`, `temporal_strategy` 제거

**실행 전 체크리스트**
- [ ] 런타임 → GPU (T4 이상) 설정
- [ ] Google Drive에 `Graph_AML/data/processed/gnn_inputs/` 아래 04번 산출물 3개 존재 확인
  - `formatted_transactions_gf.csv`
  - `formatted_transactions.csv`
  - `account_mapping.csv`
- [ ] **2번 셀** 경로 / **5번 셀** 실험 설정 확인

## 0. GPU 확인

In [ ]:
import torch

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
else:
    print("GPU가 없습니다. 런타임 유형 변경 → GPU로 전환하세요.")

## 1. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 경로 설정 ← 여기만 수정

In [ ]:
from pathlib import Path

# ============================================================
# 팀원 Drive 구조에 맞게 수정하세요
# ============================================================
DRIVE_BASE = Path("/content/drive/MyDrive/Graph_AML")
REPO_URL   = "https://github.com/JKYUPSYCHE/Graph_AML"
BRANCH     = "gnn/clustergcn"

EXPERIMENT = "GNN-01"   # 실험 그룹 이름 (예: GNN-01, GNN-02)
RUN        = "r01"      # 실행 번호 (예: r01, r02)
# ============================================================

GNN_INPUTS = DRIVE_BASE / "data" / "processed" / "gnn_inputs"
GNN_DIR    = DRIVE_BASE / "gnn"
RUN_DIR    = GNN_DIR / EXPERIMENT / RUN

for sub in ['logs', 'models', 'runs', 'feature_importance']:
    (RUN_DIR / sub).mkdir(parents=True, exist_ok=True)

required = [
    GNN_INPUTS / "formatted_transactions_gf.csv",
    GNN_INPUTS / "formatted_transactions.csv",
    GNN_INPUTS / "account_mapping.csv",
]
all_ok = True
for f in required:
    exists = f.exists()
    print(f"{'[OK]' if exists else '[MISSING]'} {f.name}")
    if not exists:
        all_ok = False

if not all_ok:
    raise FileNotFoundError("04_gnn_graph_process.ipynb 산출물이 없습니다. Drive 경로를 확인하세요.")

print("\n경로 설정 완료")
print("GNN_INPUTS :", GNN_INPUTS)
print("RUN_DIR    :", RUN_DIR)

## 3. 레포 클론 & PyG 설치

In [ ]:
import os

REPO_DIR = Path("/content/Graph_AML")

if not REPO_DIR.exists():
    os.system(f"git clone {REPO_URL} {REPO_DIR}")
    os.system(f"git -C {REPO_DIR} checkout {BRANCH}")
else:
    os.system(f"git -C {REPO_DIR} fetch origin")
    os.system(f"git -C {REPO_DIR} checkout {BRANCH}")
    os.system(f"git -C {REPO_DIR} pull origin {BRANCH}")

print("레포 준비 완료:", REPO_DIR)

In [ ]:
import torch

torch_ver = torch.__version__.split('+')[0]
cuda_tag  = 'cu' + torch.version.cuda.replace('.', '') if torch.cuda.is_available() else 'cpu'
print(f"설치 대상: torch={torch_ver}, cuda={cuda_tag}")

os.system("pip install -q torch_geometric")
os.system(f"pip install -q pyg-lib torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-{torch_ver}+{cuda_tag}.html")
os.system("pip install -q psutil tqdm tensorboard")
print("패키지 설치 완료")

## 4. 모듈 경로 설정

In [ ]:
import sys

CLUSTERGCN_DIR = REPO_DIR / "gnn" / "clustergcn"

os.chdir(CLUSTERGCN_DIR)

for p in [str(CLUSTERGCN_DIR), str(REPO_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

print("CWD        :", os.getcwd())
print("sys.path[0]:", sys.path[0])

## 5. 실험 설정 ← 여기만 수정

In [ ]:
from types import SimpleNamespace

# ============================================================
# 실험 설정 — 필요에 따라 수정
# ============================================================
args = SimpleNamespace(
    # 모델
    model      = 'pna',       # gin | gat | pna | rgcn
    data       = 'Small_HI',
    seed       = 42,

    # 학습
    n_epochs   = 100,
    patience   = 20,          # early stopping (None이면 비활성화)

    # ClusterGCN 샘플링 파라미터 (baseline의 num_neighs 대체)
    num_parts          = 300, # METIS 파티션 수 (그래프를 몇 개 클러스터로 나눌지)
    clusters_per_batch = 10,  # 배치당 합칠 파티션 수
    recursive          = True, # True: 재귀 이분법(빠름), False: 멀티레벨 k-way(정확)

    # Multi-PNA-EU 핵심 플래그
    emlps      = True,
    reverse_mp = True,
    ports      = True,
    tds        = False,
    ego        = True,

    # 저장/추론
    save_model  = True,
    unique_name = 'small_hi_epoch100_patience20_clustergcn_np300_cpb10',
    finetune    = False,
    inference   = False,

    tqdm = False,
)
# ============================================================

data_config = {
    "paths": {
        "aml_data"      : str(DRIVE_BASE / "data"),
        "gnn_inputs"    : str(GNN_INPUTS),
        "model_to_load" : str(RUN_DIR / "models"),
        "model_to_save" : str(RUN_DIR / "models"),
        "tb_log_dir"    : str(RUN_DIR / "runs"),
        "metis_cache"   : str(GNN_DIR / "metis_cache" / f"np{args.num_parts}"),
    }
}

print(f"모델      : {args.model}")
print(f"데이터    : {args.data}")
print(f"에폭      : {args.n_epochs}  |  patience: {args.patience}")
print(f"ClusterGCN: num_parts={args.num_parts}, clusters_per_batch={args.clusters_per_batch}, recursive={args.recursive}")
print(f"플래그    : emlps={args.emlps}, reverse_mp={args.reverse_mp}, ports={args.ports}, tds={args.tds}, ego={args.ego}")
print(f"저장 경로 : {RUN_DIR}")
print(f"METIS 캐시: {data_config['paths']['metis_cache']}")

## 6. 데이터 로딩

In [ ]:
import time
import logging
from util import logger_setup
from data_loading import get_data

import sys
sys.path.insert(0, str(REPO_DIR))
from utils import set_seed

LOG_DIR = RUN_DIR / 'logs'
logger_setup(log_dir=LOG_DIR, log_name=args.unique_name)
set_seed(args.seed)

logging.info("데이터 로딩 시작")
t1 = time.perf_counter()

tr_data, val_data, te_data, tr_inds, val_inds, te_inds = get_data(args, data_config)

t2 = time.perf_counter()
data_load_elapsed = t2 - t1
logging.info(f"데이터 로딩 완료: {data_load_elapsed:.2f}s")

## 7. 학습

> **Note**: ClusterGCN은 학습 시작 전 그래프 파티셔닝(METIS)을 수행합니다.  
> `num_parts=300` 기준 Small 데이터셋에서 수 분 소요될 수 있습니다.

## 6-1. METIS 파티셔닝 — tr / val / te 개별 실행 (런타임 불안정 시)

> 런타임이 자주 끊기는 경우 **6-1a → 6-1b → 6-1c** 셀을 하나씩 실행하세요.  
> 캐시가 이미 Drive에 존재하면 즉시 완료됩니다.  
> 7번 셀(학습)은 이 캐시를 자동으로 사용합니다.  
> 런타임이 안정적이면 이 셀들을 건너뛰고 7번 셀로 바로 가도 됩니다.

In [ ]:
### 6-1a. tr_data 파티셔닝
import os
from torch_geometric.loader import ClusterData

cache_dir = data_config["paths"]["metis_cache"]
tr_save   = os.path.join(cache_dir, 'tr')
os.makedirs(tr_save, exist_ok=True)

cached = [f for f in os.listdir(tr_save) if f.endswith('.pt')]
if cached:
    print(f"[tr] 캐시 존재 ({len(cached)}개 파일) → 건너뜀")
else:
    print(f"[tr] METIS 파티셔닝 시작 (num_parts={args.num_parts}, recursive=True)...")
    ClusterData(tr_data, num_parts=args.num_parts, recursive=True, log=True, save_dir=tr_save)
    print("[tr] 완료 → Drive 저장됨")

In [ ]:
### 6-1b. val_data 파티셔닝
val_save = os.path.join(cache_dir, 'val')
os.makedirs(val_save, exist_ok=True)

cached = [f for f in os.listdir(val_save) if f.endswith('.pt')]
if cached:
    print(f"[val] 캐시 존재 ({len(cached)}개 파일) → 건너뜀")
else:
    print(f"[val] METIS 파티셔닝 시작 (num_parts={args.num_parts}, recursive=True)...")
    ClusterData(val_data, num_parts=args.num_parts, recursive=True, log=True, save_dir=val_save)
    print("[val] 완료 → Drive 저장됨")

In [ ]:
### 6-1c. te_data 파티셔닝
te_save = os.path.join(cache_dir, 'te')
os.makedirs(te_save, exist_ok=True)

cached = [f for f in os.listdir(te_save) if f.endswith('.pt')]
if cached:
    print(f"[te] 캐시 존재 ({len(cached)}개 파일) → 건너뜀")
else:
    print(f"[te] METIS 파티셔닝 시작 (num_parts={args.num_parts}, recursive=True)...")
    ClusterData(te_data, num_parts=args.num_parts, recursive=True, log=True, save_dir=te_save)
    print("[te] 완료 → Drive 저장됨")

In [ ]:
import psutil
import torch
from training import train_gnn

_m, _s = divmod(int(data_load_elapsed), 60)
print(f"데이터 로딩 소요 시간: {_m:02d}분 {_s:02d}초")

cpu_pct = psutil.cpu_percent(interval=1)
ram     = psutil.virtual_memory()
print(f"CPU 사용률 : {cpu_pct:.1f}%")
print(f"RAM        : {ram.used / 1024**3:.1f} / {ram.total / 1024**3:.1f} GB  ({ram.percent:.1f}%)")

if torch.cuda.is_available():
    dev   = torch.cuda.current_device()
    used  = torch.cuda.memory_allocated(dev) / 1024**3
    total = torch.cuda.get_device_properties(dev).total_memory / 1024**3
    print(f"GPU        : {torch.cuda.get_device_name(dev)}")
    print(f"GPU 메모리 : {used:.1f} / {total:.1f} GB")
else:
    print("GPU        : 없음 (CPU 모드)")

logging.info("학습 시작")
best_te, model = train_gnn(tr_data, val_data, te_data, tr_inds, val_inds, te_inds, args, data_config)

if best_te is not None:
    tn, fp, fn, tp = best_te['tn'], best_te['fp'], best_te['fn'], best_te['tp']
    total = tn + fp + fn + tp
    print("\nBest epoch test confusion matrix:")
    print(f"  TN: {tn:,} ({tn/total*100:.2f}%)  |  FP: {fp:,} ({fp/total*100:.2f}%)")
    print(f"  FN: {fn:,} ({fn/total*100:.2f}%)  |  TP: {tp:,} ({tp/total*100:.2f}%)")

## 7-1. XAI — TP/FP/FN/TN 그룹별 피처 중요도 분석 (선택)

학습 완료 후 실행하세요. 7번 셀이 먼저 실행되어 `model` 변수가 있어야 합니다.

- TP/FP/FN/TN 각 그룹에서 최대 `n_samples`개 샘플링
- k-hop 서브그래프 고정 후 gradient saliency로 피처 중요도 계산
- `{unique_name}_feature_importance.csv` : 그룹별 평균·표준편차
- `{unique_name}_feature_importance_individual.csv` : 개별 샘플 결과

> **Note**: ClusterGCN XAI는 그래프 파티셔닝을 재수행하므로 시작까지 수 분 소요될 수 있습니다.

In [ ]:
from xai import run_xai
from train_util import get_loaders
import torch

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# train_gnn 내부에서 add_arange_ids가 이미 적용된 상태
# XAI용 te_loader 재생성 (METIS 캐시 사용)
cache_dir = data_config["paths"].get("metis_cache", None)
_, _, xai_te_loader = get_loaders(tr_data, val_data, te_data, args, cache_dir=cache_dir)

XAI_OUT = RUN_DIR / 'feature_importance'

summary_df, records_df = run_xai(
    te_loader  = xai_te_loader,
    te_inds    = te_inds,
    model      = model,
    te_data    = te_data,
    device     = device,
    args       = args,
    out_dir    = XAI_OUT,
    run_name   = args.unique_name,
    n_samples  = 500,
)

print("\n그룹별 평균 피처 중요도:")
mean_cols = [c for c in summary_df.columns if c.endswith('__mean')]
display(summary_df[mean_cols].round(4))

print(f"\n저장 경로: {XAI_OUT}")
print(f"  - {args.unique_name}_feature_importance.csv           (그룹별 평균·표준편차)")
print(f"  - {args.unique_name}_feature_importance_individual.csv (개별 샘플 결과)")

## 8. TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {RUN_DIR}/runs

## 9. 추론 (선택)

저장된 모델로 test set 추론만 할 경우 실행하세요.  
`args.unique_name`이 저장 시 이름과 일치해야 합니다.  
실행 순서: **5번 셀 → 6번 셀 → 9번 셀**

In [ ]:
from inference import infer_gnn

infer_gnn(tr_data, val_data, te_data, tr_inds, val_inds, te_inds, args, data_config)